## Import Libraries

In [1]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pyodbc

In [2]:
# nltk.download('vader_lexicon')

## Fetch Data From SQL

In [3]:
def fetch_data_from_sql():
    conn_str = (
    "Driver={SQL Server};"
    "Server=localhost\\SQLEXPRESS;"
    "Database=PortfolioProject_MarketingAnalytics;"
    "Trusted_Connection=yes;"
    )

    conn = pyodbc.connect(conn_str)

    query = """
    SELECT 
        ReviewID,
        CustomerID,
        ProductID,
        ReviewDate,
        Rating,
        ReviewText
    FROM customer_reviews
    """


    df = pd.read_sql(query,conn)
    conn.close()
    return df

In [4]:
df = fetch_data_from_sql()
df.head()

C:\Users\Dell\AppData\Local\Temp\ipykernel_16308\3235099261.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query,conn)


,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"Average experience, nothing special."
1,2,80,19,2024-12-25,5,The quality is top-notch.
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"Average experience, nothing special."


In [5]:
sia = SentimentIntensityAnalyzer()

## Calculate sentiment score using VADER

In [6]:
def calculate_sentiment(review):
    if pd.isna(review):
        return 0
    sentiment = sia.polarity_scores(str(review))
    return sentiment['compound']

## Defile a function to categorize the Sentiment using both sentiment score and Review Rating


In [7]:
def categorize_sentiment(score,rating):
    # using both sentiment score and review rating to determine the dentiment category
    if score > 0.05: # Positive Sentiment
        if rating >= 4:
            return 'Positive' 
        elif rating == 3:
            return 'Mixed Positive'
        else:
            return 'Mixed Negative'
    elif score < -0.05: # Negative sentiment
        if rating <= 2:
            return 'Negative'
        elif rating == 3:
            return 'Mixed Negative'
        else :
            return 'Mixed Positive'
    else: # Neutral Sentiment Score
        if rating >= 4:
            return 'Positive' 
        elif rating <= 2 :
            return 'Negative'
        else:
            return 'Neutral'

## Define a function to bucket sentiment scores into Rating text range

In [8]:
def sentiment_bucket(score):
    if score >= 0.5:
        return '0.5 to 1'
    elif 0.0 <= score <=0.49:
        return '0.0 to 0.49'
    elif -0.5 <= score < 0.00 :
        return '-0.5 to 0.00'
    else :
        return '-1.0 to -0.5' 

## Apply all function

In [9]:
# Apply sentiment Analysis to calculate sentiment Scores for each review
df["sentimentScore"] = df["ReviewText"].apply(calculate_sentiment)

# Apply Sentiment Bucketing fxn to categorize scores into different Ranges 
df['sentimentBucket'] = df['sentimentScore'].apply(sentiment_bucket)

# Apply sentiment Categorization using both Text and Rating
df['sentimentCategory'] = df.apply(lambda row:categorize_sentiment(row['sentimentScore'],row['Rating']),axis=1)

In [14]:
pd.set_option('display.max_columns',10)
df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,sentimentScore,sentimentBucket,sentimentCategory
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089,-0.5 to 0.00,Mixed Negative
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000,0.0 to 0.49,Positive
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000,0.0 to 0.49,Positive
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382,0.0 to 0.49,Mixed Positive
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089,-0.5 to 0.00,Mixed Negative


In [12]:
df.to_csv('fact_sentiment.csv',index=False)

In [17]:
df.set_index('ReviewID',inplace = True)

In [18]:
df

,CustomerID,ProductID,ReviewDate,Rating,ReviewText,sentimentScore,sentimentBucket,sentimentCategory
ReviewID,,,,,,,,
1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089,-0.5 to 0.00,Mixed Negative
2,80,19,2024-12-25,5,The quality is top-notch.,0.0000,0.0 to 0.49,Positive
3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000,0.0 to 0.49,Positive
4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382,0.0 to 0.49,Mixed Positive
5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089,-0.5 to 0.00,Mixed Negative
...,...,...,...,...,...,...,...,...
1359,28,4,2023-05-25,3,Not worth the money.,-0.1695,-0.5 to 0.00,Mixed Negative
1360,58,12,2023-11-13,2,"Average experience, nothing special.",-0.3089,-0.5 to 0.00,Negative
1361,96,15,2023-03-07,5,Customer support was very helpful.,0.6997,0.5 to 1,Positive
